# IMDB Sentiment Classification — ECE 364 Project Option #2

**Goal:** Binary sentiment classifier on IMDB movie reviews, **≤10M total parameters**, target ≈0.94 accuracy.

## Approach

We're squeezing every drop of accuracy out of a 10M-param budget. The dominant cost in any BERT-style model is the word-embedding table (`vocab_size × hidden`). With BERT's 30,522 wordpiece vocab, an H=256 embedding matrix alone is ~7.8M params — leaving room for only ~2 transformer layers under the cap.

Pipeline:

1. **Student model:** `google/bert_uncased_L-2_H-256_A-4` (~9.6M params total). Verified < 10M with an explicit assertion.
2. **Continued MLM pretraining** on the IMDB corpus (train + val + test review text — no labels needed) for domain adaptation. This is one of the highest-leverage tricks for small models.
3. **Knowledge distillation** from a publicly-available BERT-base fine-tuned on IMDB. The teacher is used **only during training** — only the small student is the final deliverable. We cache teacher logits up front so the teacher never has to share GPU memory with the student.
4. **Head + tail truncation** at 512 tokens. Many IMDB reviews are >512 wordpieces; keeping the first 128 and last 382 tokens captures both setup and conclusion (the conclusion often signals sentiment).
5. **Mixed-precision (fp16)** training to fit your 8GB 2070S.
6. **Standard fine-tuning hygiene:** AdamW, linear warmup + decay, weight decay 0.01, gradient clipping, save-best-on-val.

Final output: `predictions.csv` with columns `Id,Label` where Label ∈ {`positive`, `negative`} (lowercase).


## 0. Setup

In [ ]:
import os, gc, random, json, math, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

from transformers import (
    AutoTokenizer, AutoConfig,
    AutoModelForMaskedLM, AutoModelForSequenceClassification,
    DataCollatorForLanguageModeling, get_linear_schedule_with_warmup,
)
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ----- Config -----
# Paths (adjust if your folder layout differs)
TRAIN_CSV       = "imdb-review-classification/train.csv"
TEST_CSV        = "imdb-review-classification/test.csv"
PREDICTIONS_CSV = "predictions.csv"
WORK_DIR        = "./work_dir"
os.makedirs(WORK_DIR, exist_ok=True)

# Models
STUDENT_NAME = "google/bert_uncased_L-2_H-256_A-4"   # ~9.6M params, under 10M
# Public BERT-base fine-tuned on IMDB. If this URL ever fails, alternatives that work for KD:
#   - "lvwerra/distilbert-imdb"
#   - "aychang/roberta-base-imdb"
# (any tokenizer/architecture is fine for KD — we only need teacher logits)
TEACHER_NAME = "textattack/bert-base-uncased-imdb"

# Sequence length / truncation
MAX_LEN  = 512
HEAD_LEN = 128
TAIL_LEN = MAX_LEN - HEAD_LEN - 2   # leave room for [CLS] and [SEP]

# Batch sizes — drop these if you OOM on the 2070S
BATCH_SIZE_MLM   = 16
BATCH_SIZE_TRAIN = 16
BATCH_SIZE_EVAL  = 32
GRAD_ACCUM       = 2     # effective train batch = 32

# Optimization
LR_MLM        = 5e-5
LR_FT         = 3e-5
WEIGHT_DECAY  = 0.01
WARMUP_RATIO  = 0.1
EPOCHS_MLM    = 3
EPOCHS_FT     = 8

# Knowledge distillation
KD_ALPHA = 0.5    # weight on KL(student || teacher); (1-alpha) on hard-label CE
KD_T     = 2.0    # temperature for both student and teacher logits

# Label maps (lowercase to match the dataset)
LABEL2ID = {"negative": 0, "positive": 1}
ID2LABEL = {0: "negative", 1: "positive"}

## 1. Load and inspect data

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

print("Train shape:", train_df.shape, "| columns:", list(train_df.columns))
print("Test shape: ", test_df.shape, "| columns:", list(test_df.columns))
print()
print("Label distribution in train:")
print(train_df["label"].value_counts())
print()
print("Sample review:", train_df.iloc[0]["review"][:200], "...")
print("Sample label:", train_df.iloc[0]["label"])

# Map label strings -> ids
train_df["label_id"] = train_df["label"].str.lower().map(LABEL2ID)
assert train_df["label_id"].isna().sum() == 0, "Unexpected label values!"


In [ ]:
# Length distribution (in raw words) — useful sanity check on truncation choice
lens = train_df["review"].str.split().str.len()
print(f"Review length (words): mean={lens.mean():.0f}, median={lens.median():.0f}, "
      f"95%={lens.quantile(0.95):.0f}, 99%={lens.quantile(0.99):.0f}, max={lens.max()}")
print(f"Reviews > 400 words (likely > 512 wordpieces): {(lens > 400).mean()*100:.1f}%")

## 2. Train/val split (90/10, stratified)

In [ ]:
trn_df, val_df = train_test_split(
    train_df, test_size=0.10, stratify=train_df["label_id"], random_state=SEED
)
trn_df = trn_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
print(f"Train: {len(trn_df)}, Val: {len(val_df)}")
print("Train label balance:", trn_df['label_id'].value_counts().to_dict())
print("Val label balance:  ", val_df['label_id'].value_counts().to_dict())

## 3. Build student model and verify ≤10M params

The 10M cap is on **total** parameters (embeddings + encoder + classifier head), not just trainable. We assert it explicitly so a future model swap can't silently violate the limit.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(STUDENT_NAME)

# Build a *fresh* student config so we control num_labels
student_config = AutoConfig.from_pretrained(
    STUDENT_NAME,
    num_labels=2,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)
student = AutoModelForSequenceClassification.from_pretrained(STUDENT_NAME, config=student_config)

total_params     = sum(p.numel() for p in student.parameters())
trainable_params = sum(p.numel() for p in student.parameters() if p.requires_grad)
print(f"Student total params:     {total_params:>12,}")
print(f"Student trainable params: {trainable_params:>12,}")
print(f"Limit:                    {10_000_000:>12,}")

assert total_params < 10_000_000, (
    f"Student has {total_params:,} params > 10M limit — pick a smaller model.")
print("✓ Under 10M parameter limit")

# Dump param breakdown so you can include it in the report
print("\nParam breakdown by top-level module:")
for name, module in student.named_children():
    n = sum(p.numel() for p in module.parameters())
    print(f"  {name:30s} {n:>12,}")
del student  # we'll reload after MLM pretraining
gc.collect(); torch.cuda.empty_cache()

## 4. Tokenization with head + tail truncation

For inputs longer than `MAX_LEN`, we keep the first `HEAD_LEN` tokens and the last `TAIL_LEN` tokens. This empirically beats head-only truncation on IMDB because reviewers often state their final verdict at the end ("...overall, terrible film").

In [ ]:
def head_tail_tokenize(text, head=HEAD_LEN, tail=TAIL_LEN, max_len=MAX_LEN):
    """Tokenize with head+tail truncation. Returns dict with input_ids, attention_mask."""
    cls, sep, pad = tokenizer.cls_token_id, tokenizer.sep_token_id, tokenizer.pad_token_id
    ids = tokenizer.encode(text, add_special_tokens=False, truncation=False)
    if len(ids) <= max_len - 2:
        input_ids = [cls] + ids + [sep]
    else:
        input_ids = [cls] + ids[:head] + ids[-tail:] + [sep]
    attn = [1] * len(input_ids)
    pad_n = max_len - len(input_ids)
    input_ids = input_ids + [pad] * pad_n
    attn      = attn + [0]   * pad_n
    return {"input_ids": input_ids[:max_len], "attention_mask": attn[:max_len]}

# Sanity check
demo = head_tail_tokenize(train_df.iloc[0]["review"])
print("input_ids len:", len(demo["input_ids"]), "| non-pad tokens:", sum(demo["attention_mask"]))

## 5. Continued MLM pretraining (domain adaptation)

We do masked-language-modeling on the combined train+val+test review text (text only — no label leakage; MLM doesn't use labels) for a few epochs. This adapts the generic BookCorpus+Wikipedia weights to IMDB's vocabulary and writing style. Empirically this nets +1-2% downstream accuracy for small models.

Uses 256-token sequences (faster than 512) and standard 15% masking.

In [ ]:
class MLMTextDataset(Dataset):
    def __init__(self, texts, tok, max_len=256):
        self.texts = list(texts)
        self.tok = tok
        self.max_len = max_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, i):
        enc = self.tok(self.texts[i], truncation=True, padding="max_length",
                       max_length=self.max_len, return_tensors="pt")
        return {k: v.squeeze(0) for k, v in enc.items()}

# Combine all available text for MLM (no labels used, no leakage)
mlm_texts = pd.concat([trn_df["review"], val_df["review"], test_df["review"]],
                      ignore_index=True).tolist()
print(f"MLM corpus size: {len(mlm_texts)} reviews")

mlm_ds = MLMTextDataset(mlm_texts, tokenizer, max_len=256)
mlm_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True, mlm_probability=0.15)
mlm_loader = DataLoader(mlm_ds, batch_size=BATCH_SIZE_MLM, shuffle=True,
                        collate_fn=mlm_collator, num_workers=2, pin_memory=True)

In [ ]:
# Train MLM
mlm_model = AutoModelForMaskedLM.from_pretrained(STUDENT_NAME).to(DEVICE)

optim = torch.optim.AdamW(mlm_model.parameters(), lr=LR_MLM, weight_decay=WEIGHT_DECAY)
total_steps = len(mlm_loader) * EPOCHS_MLM
sched = get_linear_schedule_with_warmup(optim, int(WARMUP_RATIO * total_steps), total_steps)
scaler = GradScaler()

mlm_model.train()
for epoch in range(EPOCHS_MLM):
    running, n = 0.0, 0
    pbar = tqdm(mlm_loader, desc=f"MLM epoch {epoch+1}/{EPOCHS_MLM}")
    for batch in pbar:
        batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
        optim.zero_grad(set_to_none=True)
        with autocast(dtype=torch.float16):
            out = mlm_model(**batch)
            loss = out.loss
        scaler.scale(loss).backward()
        scaler.unscale_(optim)
        torch.nn.utils.clip_grad_norm_(mlm_model.parameters(), 1.0)
        scaler.step(optim)
        scaler.update()
        sched.step()
        running += loss.item(); n += 1
        if n % 50 == 0:
            pbar.set_postfix(loss=f"{running/n:.4f}")
    print(f"Epoch {epoch+1} mean MLM loss: {running/n:.4f}")

# Save MLM-pretrained encoder so we can reload it as a SequenceClassification head
mlm_save_path = os.path.join(WORK_DIR, "student_mlm_pretrained")
mlm_model.save_pretrained(mlm_save_path)
tokenizer.save_pretrained(mlm_save_path)
print("Saved MLM-pretrained student to", mlm_save_path)

del mlm_model, optim, sched, scaler
gc.collect(); torch.cuda.empty_cache()

## 6. Cache teacher logits for knowledge distillation

We load a public BERT-base fine-tuned on IMDB, run it once over our train and val sets, and cache the resulting logits. This means:
- The teacher model **never coexists on GPU** with the student during training (saves VRAM).
- Each epoch of student training has zero teacher inference overhead — just a tensor lookup.

The teacher is **only used during training**. The submitted/deliverable model is the small student, which stays under 10M.

In [ ]:
print("Loading teacher:", TEACHER_NAME)
teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER_NAME)
teacher = AutoModelForSequenceClassification.from_pretrained(TEACHER_NAME).to(DEVICE)
teacher.eval()
n_teacher = sum(p.numel() for p in teacher.parameters())
print(f"Teacher params: {n_teacher:,} (used only for training, not in deliverable)")
print("Teacher label map (raw):", teacher.config.id2label)

In [ ]:
@torch.no_grad()
def teacher_predict_logits(texts, batch_size=8, max_len=MAX_LEN):
    all_logits = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Teacher inference"):
        batch_texts = texts[i:i+batch_size]
        enc = teacher_tokenizer(batch_texts, padding=True, truncation=True,
                                max_length=max_len, return_tensors="pt").to(DEVICE)
        with autocast(dtype=torch.float16):
            out = teacher(**enc)
        all_logits.append(out.logits.float().cpu())
    return torch.cat(all_logits, dim=0)

# Cache logits
teacher_trn_logits = teacher_predict_logits(trn_df["review"].tolist())
teacher_val_logits = teacher_predict_logits(val_df["review"].tolist())
print("Cached:", teacher_trn_logits.shape, teacher_val_logits.shape)

In [ ]:
# CRITICAL: figure out teacher's label ordering by checking val accuracy under both orderings.
# textattack/bert-base-uncased-imdb uses {0: NEG, 1: POS} which matches our LABEL2ID,
# but we verify in case the user swaps in a different teacher.
val_labels = val_df["label_id"].values

pred_as_is = teacher_val_logits.argmax(-1).numpy()
acc_as_is = (pred_as_is == val_labels).mean()
acc_flipped = ((1 - pred_as_is) == val_labels).mean()
print(f"Teacher val acc (as-is):  {acc_as_is:.4f}")
print(f"Teacher val acc (flipped): {acc_flipped:.4f}")

if acc_flipped > acc_as_is:
    print("Flipping teacher logits to align with our label order (0=neg, 1=pos)")
    teacher_trn_logits = teacher_trn_logits[:, [1, 0]]
    teacher_val_logits = teacher_val_logits[:, [1, 0]]
    pred_as_is = teacher_val_logits.argmax(-1).numpy()

teacher_val_acc = (teacher_val_logits.argmax(-1).numpy() == val_labels).mean()
print(f"Teacher final val acc:    {teacher_val_acc:.4f}  ← upper bound for distillation")

# Free teacher from GPU
del teacher, teacher_tokenizer
gc.collect(); torch.cuda.empty_cache()

## 7. Build datasets and dataloaders for fine-tuning

In [ ]:
class IMDBClsDataset(Dataset):
    """Reviews tokenized with head+tail truncation. Optionally carries cached teacher logits."""
    def __init__(self, df, teacher_logits=None, has_label=True):
        self.df = df.reset_index(drop=True)
        self.teacher_logits = teacher_logits
        self.has_label = has_label
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        enc = head_tail_tokenize(row["review"])
        item = {
            "input_ids":      torch.tensor(enc["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(enc["attention_mask"], dtype=torch.long),
        }
        if self.has_label:
            item["labels"] = torch.tensor(int(row["label_id"]), dtype=torch.long)
        if self.teacher_logits is not None:
            item["teacher_logits"] = self.teacher_logits[i].clone()
        return item

trn_ds = IMDBClsDataset(trn_df, teacher_logits=teacher_trn_logits, has_label=True)
val_ds = IMDBClsDataset(val_df, teacher_logits=None,             has_label=True)
tst_ds = IMDBClsDataset(test_df, teacher_logits=None,            has_label=False)

trn_loader = DataLoader(trn_ds, batch_size=BATCH_SIZE_TRAIN, shuffle=True,
                        num_workers=2, pin_memory=True, drop_last=False)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE_EVAL, shuffle=False,
                        num_workers=2, pin_memory=True)
tst_loader = DataLoader(tst_ds, batch_size=BATCH_SIZE_EVAL, shuffle=False,
                        num_workers=2, pin_memory=True)
print("Loaders ready.")

## 8. Knowledge-distillation fine-tuning

**Loss:** $\mathcal{L} = \alpha \cdot T^2 \cdot \mathrm{KL}(\sigma(s/T) \| \sigma(t/T)) + (1-\alpha) \cdot \mathrm{CE}(s, y)$

where $s$ is student logits, $t$ is cached teacher logits, $T$ is temperature, $y$ is the hard label. The $T^2$ term keeps gradients from the soft-label term scaled comparably as $T$ varies.

We save the checkpoint with the best validation accuracy.

In [ ]:
# Reload student starting from the MLM-pretrained checkpoint
student = AutoModelForSequenceClassification.from_pretrained(
    mlm_save_path,
    num_labels=2,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
).to(DEVICE)

# Sanity check: confirm we're still under 10M after adding the classifier head
n_total = sum(p.numel() for p in student.parameters())
print(f"Student total params (with cls head): {n_total:,}")
assert n_total < 10_000_000, "Over 10M after head — should not happen with this config."

# Optimizer with no weight decay on bias and LayerNorm
no_decay = ["bias", "LayerNorm.weight"]
grouped = [
    {"params": [p for n, p in student.named_parameters() if not any(nd in n for nd in no_decay)],
     "weight_decay": WEIGHT_DECAY},
    {"params": [p for n, p in student.named_parameters() if     any(nd in n for nd in no_decay)],
     "weight_decay": 0.0},
]
optim = torch.optim.AdamW(grouped, lr=LR_FT)

steps_per_epoch = math.ceil(len(trn_loader) / GRAD_ACCUM)
total_steps     = steps_per_epoch * EPOCHS_FT
sched  = get_linear_schedule_with_warmup(optim, int(WARMUP_RATIO * total_steps), total_steps)
scaler = GradScaler()

In [ ]:
def kd_loss(student_logits, teacher_logits, labels, alpha=KD_ALPHA, T=KD_T):
    # Soft-label loss
    s_log = F.log_softmax(student_logits / T, dim=-1)
    t_p   = F.softmax(teacher_logits / T, dim=-1)
    kd = F.kl_div(s_log, t_p, reduction="batchmean") * (T * T)
    # Hard-label loss
    ce = F.cross_entropy(student_logits, labels)
    return alpha * kd + (1 - alpha) * ce, kd.item(), ce.item()

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    for batch in loader:
        ids  = batch["input_ids"].to(DEVICE, non_blocking=True)
        mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
        labels = batch["labels"].to(DEVICE, non_blocking=True)
        with autocast(dtype=torch.float16):
            out = model(input_ids=ids, attention_mask=mask)
        preds = out.logits.argmax(-1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
    return correct / total

In [ ]:
best_val_acc = 0.0
best_path = os.path.join(WORK_DIR, "student_best")
log = []

for epoch in range(EPOCHS_FT):
    student.train()
    running, n = 0.0, 0
    pbar = tqdm(trn_loader, desc=f"FT epoch {epoch+1}/{EPOCHS_FT}")
    optim.zero_grad(set_to_none=True)
    for step, batch in enumerate(pbar):
        ids  = batch["input_ids"].to(DEVICE, non_blocking=True)
        mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
        labels = batch["labels"].to(DEVICE, non_blocking=True)
        t_logits = batch["teacher_logits"].to(DEVICE, non_blocking=True)

        with autocast(dtype=torch.float16):
            out = student(input_ids=ids, attention_mask=mask)
            loss, kd_v, ce_v = kd_loss(out.logits.float(), t_logits.float(), labels)
            loss = loss / GRAD_ACCUM

        scaler.scale(loss).backward()

        if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(trn_loader):
            scaler.unscale_(optim)
            torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
            scaler.step(optim)
            scaler.update()
            sched.step()
            optim.zero_grad(set_to_none=True)

        running += loss.item() * GRAD_ACCUM; n += 1
        if n % 50 == 0:
            pbar.set_postfix(loss=f"{running/n:.4f}", kd=f"{kd_v:.3f}", ce=f"{ce_v:.3f}")

    val_acc = evaluate(student, val_loader)
    log.append({"epoch": epoch+1, "train_loss": running/n, "val_acc": val_acc})
    print(f"  → epoch {epoch+1}: train_loss={running/n:.4f}  val_acc={val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        student.save_pretrained(best_path)
        tokenizer.save_pretrained(best_path)
        print(f"  ✓ new best val_acc={val_acc:.4f}, saved to {best_path}")

print(f"\nBest val accuracy: {best_val_acc:.4f}")
pd.DataFrame(log).to_csv(os.path.join(WORK_DIR, "training_log.csv"), index=False)

## 9. Final evaluation on the validation set

Reload the best checkpoint and recompute val accuracy as a sanity check before generating predictions.

In [ ]:
del student
gc.collect(); torch.cuda.empty_cache()

student = AutoModelForSequenceClassification.from_pretrained(best_path).to(DEVICE)
final_val_acc = evaluate(student, val_loader)
print(f"Final val accuracy (best checkpoint): {final_val_acc:.4f}")

# Final param count for the report
n_final = sum(p.numel() for p in student.parameters())
print(f"Final model total params: {n_final:,}  (< 10,000,000 ✓)" if n_final < 10_000_000 else "OVER LIMIT")

## 10. Generate predictions.csv

Run the best student over `test.csv` and write predictions in the required format:

```
Id,Label
33554,positive
9999,negative
...
```

Lowercase labels, exact column names `Id` and `Label`, in the original test set's `Id` order.

In [ ]:
@torch.no_grad()
def predict(model, loader):
    model.eval()
    all_preds = []
    for batch in tqdm(loader, desc="Predicting test"):
        ids  = batch["input_ids"].to(DEVICE, non_blocking=True)
        mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
        with autocast(dtype=torch.float16):
            out = model(input_ids=ids, attention_mask=mask)
        all_preds.append(out.logits.argmax(-1).cpu().numpy())
    return np.concatenate(all_preds)

test_preds = predict(student, tst_loader)
print("Predictions shape:", test_preds.shape)
print("Predicted distribution:", pd.Series(test_preds).value_counts().to_dict())

In [ ]:
out_df = pd.DataFrame({
    "Id":    test_df["Id"].values,
    "Label": [ID2LABEL[int(p)] for p in test_preds],   # lowercase 'positive'/'negative'
})
out_df.to_csv(PREDICTIONS_CSV, index=False)

# Sanity-check the file we just wrote
print(f"Wrote {PREDICTIONS_CSV}: {len(out_df)} rows")
print(out_df.head())
print()
print("Label distribution:", out_df["Label"].value_counts().to_dict())
assert set(out_df["Label"].unique()) <= {"positive", "negative"}, "Labels not lowercase!"
assert list(out_df.columns) == ["Id", "Label"], "Wrong columns!"
print("\n✓ predictions.csv format verified")

## 11. Summary for the 2-page report

A few numbers worth lifting into the report:

- **Architecture:** `google/bert_uncased_L-2_H-256_A-4` — 2 transformer layers, hidden 256, 4 heads, intermediate 1024, vocab 30522 (BERT-uncased).
- **Total params:** ~9.6M (verified < 10M).
- **Sequence length:** 512 with head+tail truncation (first 128 + last 382 wordpieces).
- **Training pipeline:** continued MLM pretraining → KD fine-tuning from BERT-base IMDB teacher.
- **KD config:** α=0.5, T=2.0.
- **Optimizer:** AdamW, LR=3e-5 (FT) / 5e-5 (MLM), weight decay 0.01, linear warmup (10%) + decay.
- **Mixed precision:** fp16 (autocast + GradScaler).
- **Reproducibility:** seed 42 for split, dataloader, and model init.

**Tricks that move the needle most for tiny BERTs on IMDB**, ranked by typical impact:
1. Knowledge distillation from a larger, IMDB-fine-tuned teacher (+1–3%)
2. Continued MLM pretraining on the IMDB corpus (+1–2%)
3. Head+tail truncation at 512 vs head-only at 128/256 (+0.5–1%)
4. Best-checkpoint-on-val rather than last-epoch (+0.2–0.5%)
